In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from urllib.parse import urlparse

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
CSV_PATH = Path("archive/Myntra Fasion Clothing.csv")

if not CSV_PATH.exists():
    CSV_PATH = Path("Myntra Fasion Clothing.csv")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        "CSV file not found. Check the CSV_PATH variable "
        "and enter the correct file location."
    )

df = pd.read_csv(CSV_PATH, low_memory=False)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Number of products:", len(df))

Dataset loaded successfully!
Dataset shape: (526564, 13)
Number of products: 526564


In [3]:
display(df.head())

,URL,Product_id,BrandName,Category,Individual_category,category_by_Gender,Description,DiscountPrice (in Rs),OriginalPrice (in Rs),DiscountOffer,SizeOption,Ratings,Reviews
0,https://www.myntra.com/jeans/roadster/roadster...,2296012,Roadster,Bottom Wear,jeans,Men,roadster men navy blue slim fit mid rise clean...,824.0,1499.0,45% OFF,"28, 30, 32, 34, 36",3.9,999.0
1,https://www.myntra.com/track-pants/locomotive/...,13780156,LOCOMOTIVE,Bottom Wear,track-pants,Men,locomotive men black white solid slim fit tra...,517.0,1149.0,55% OFF,"S, M, L, XL",4.0,999.0
2,https://www.myntra.com/shirts/roadster/roadste...,11895958,Roadster,Topwear,shirts,Men,roadster men navy white black geometric print...,629.0,1399.0,55% OFF,"38, 40, 42, 44, 46, 48",4.3,999.0
3,https://www.myntra.com/shapewear/zivame/zivame...,4335679,Zivame,Lingerie & Sleep Wear,shapewear,Women,zivame women black saree shapewear zi3023core0...,893.0,1295.0,31% OFF,"S, M, L, XL, XXL",4.2,999.0
4,https://www.myntra.com/tshirts/roadster/roadst...,11690882,Roadster,Western,tshirts,Women,roadster women white solid v neck pure cotton ...,NaN,599.0,35% OFF,"XS, S, M, L, XL",4.2,999.0


In [4]:
print("Available columns:")

for column in df.columns:
    print(column)

Available columns:
URL
Product_id
BrandName
Category
Individual_category
category_by_Gender
Description
DiscountPrice (in Rs)
OriginalPrice (in Rs)
DiscountOffer
SizeOption
Ratings
Reviews


In [5]:
data_quality = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (
        df.isna().mean() * 100
    ).round(2)
})

display(data_quality)

,Data Type,Missing Values,Missing Percentage
URL,object,0,0.00
Product_id,int64,0,0.00
BrandName,object,0,0.00
Category,object,0,0.00
Individual_category,object,0,0.00
category_by_Gender,object,0,0.00
Description,object,0,0.00
DiscountPrice (in Rs),float64,193158,36.68
OriginalPrice (in Rs),float64,0,0.00
DiscountOffer,object,74306,14.11


In [6]:
print("Duplicate rows:", df.duplicated().sum())

if "Product_id" in df.columns:
    print(
        "Duplicate Product IDs:",
        df["Product_id"].duplicated().sum()
    )

if "URL" in df.columns:
    print("Missing URLs:", df["URL"].isna().sum())
    print("Duplicate URLs:", df["URL"].duplicated().sum())

    display(df[["URL", "Product_id", "Description"]].head(10))

Duplicate rows: 0
Duplicate Product IDs: 0
Missing URLs: 0
Duplicate URLs: 0


,URL,Product_id,Description
0,https://www.myntra.com/jeans/roadster/roadster...,2296012,roadster men navy blue slim fit mid rise clean...
1,https://www.myntra.com/track-pants/locomotive/...,13780156,locomotive men black white solid slim fit tra...
2,https://www.myntra.com/shirts/roadster/roadste...,11895958,roadster men navy white black geometric print...
3,https://www.myntra.com/shapewear/zivame/zivame...,4335679,zivame women black saree shapewear zi3023core0...
4,https://www.myntra.com/tshirts/roadster/roadst...,11690882,roadster women white solid v neck pure cotton ...
5,https://www.myntra.com/tops/mast--harbour/mast...,2490950,mast harbour women yellow solid tank top
6,https://www.myntra.com/trousers/highlander/hig...,6744434,highlander men olive green slim fit solid regu...
7,https://www.myntra.com/tops/mayra/mayra-pink-e...,8439415,mayra pink embroidered a line pure cotton top
8,https://www.myntra.com/tshirts/roadster/roadst...,17381394,roadster women pack of 2 solid t shirts
9,https://www.myntra.com/jeans/herenow/herenow-m...,2359257,herenow men blue slim fit mid rise clean look ...


In [7]:
for column in [
    "BrandName",
    "Category",
    "Individual_category",
    "category_by_Gender"
]:
    if column in df.columns:
        print(f"\nTop values in {column}:")
        display(df[column].value_counts().head(10))


Top values in BrandName:


BrandName
Pothys                   16005
Roadster                 10935
KALINI                    9589
HERE&NOW                  6515
HRX by Hrithik Roshan     5297
Mast & Harbour            5148
Ethnic basket             4790
Mitera                    4695
Puma                      4679
max                       4650
Name: count, dtype: int64


Top values in Category:


Category
Indian Wear                 145845
Western                     140992
Topwear                      74537
Bottom Wear                  55439
Lingerie & Sleep Wear        55258
Sports Wear                  20627
Inner Wear &  Sleep Wear     20370
Plus Size                    13496
Name: count, dtype: int64


Top values in Individual_category:


Individual_category
tshirts       61198
sarees        57915
tops          39126
kurtas        38984
dresses       35590
shirts        32692
kurta-sets    26592
jeans         25206
trousers      24706
bra           18097
Name: count, dtype: int64


Top values in category_by_Gender:


category_by_Gender
Women    339185
Men      187379
Name: count, dtype: int64

In [8]:
feature_columns = [
    "BrandName",
    "Category",
    "Individual_category",
    "category_by_Gender",
    "Description"
]

missing_columns = [
    column for column in feature_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("All required feature columns are available.")

All required feature columns are available.


In [9]:
df["features"] = (
    df[feature_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
    .str.strip()
)

display(
    df[["Product_id", "BrandName", "Category", "features"]]
    .head(5)
)

,Product_id,BrandName,Category,features
0,2296012,Roadster,Bottom Wear,roadster bottom wear jeans men roadster men na...
1,13780156,LOCOMOTIVE,Bottom Wear,locomotive bottom wear track-pants men locomot...
2,11895958,Roadster,Topwear,roadster topwear shirts men roadster men navy ...
3,4335679,Zivame,Lingerie & Sleep Wear,zivame lingerie & sleep wear shapewear women z...
4,11690882,Roadster,Western,roadster western tshirts women roadster women ...


In [10]:
vectorizer = TfidfVectorizer(
    max_features=10_000,
    min_df=2,
    stop_words="english",
    dtype=np.float32
)

tfidf_matrix = vectorizer.fit_transform(df["features"])

matrix_memory_mb = (
    tfidf_matrix.data.nbytes
    + tfidf_matrix.indices.nbytes
    + tfidf_matrix.indptr.nbytes
) / (1024 ** 2)

print("TF-IDF completed successfully!")
print("Number of products:", tfidf_matrix.shape[0])
print("Number of text features:", tfidf_matrix.shape[1])
print("Matrix format:", tfidf_matrix.format)
print("Approximate matrix memory:", round(matrix_memory_mb, 2), "MB")

TF-IDF completed successfully!
Number of products: 526564
Number of text features: 9011
Matrix format: csr
Approximate matrix memory: 44.94 MB


In [11]:
def recommend_similar(
    product_id,
    top_n=5,
    exclude_exact_duplicates=True
):
    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    matches = df.index[df["Product_id"] == product_id]

    if len(matches) == 0:
        print(f"Product ID {product_id} was not found.")
        return pd.DataFrame()

    # Use the first matching product
    idx = matches[0]

    # Calculate similarity against all products
    scores = linear_kernel(
        tfidf_matrix[idx],
        tfidf_matrix
    ).ravel()

    # Do not recommend the selected product itself
    scores[idx] = -1

    # Optionally exclude products with identical text features
    if exclude_exact_duplicates:
        same_features = (
            df["features"]
            .eq(df.iloc[idx]["features"])
            .to_numpy()
        )
        scores[same_features] = -1

    # Sort products by similarity, highest first
    ranked_indices = np.argsort(scores)[::-1]

    # Keep only valid recommendations
    ranked_indices = [
        i for i in ranked_indices
        if scores[i] >= 0
    ][:top_n]

    result_columns = [
        "Product_id",
        "BrandName",
        "Category",
        "Individual_category",
        "category_by_Gender",
        "Description",
        "URL"
    ]

    results = df.iloc[ranked_indices][result_columns].copy()

    results["similarity_score"] = scores[ranked_indices]

    return results.reset_index(drop=True)

In [12]:
test_product_id = df["Product_id"].dropna().iloc[0]

print("Selected Product ID:", test_product_id)

selected_product = df.loc[
    df["Product_id"] == test_product_id,
    [
        "Product_id",
        "BrandName",
        "Category",
        "Individual_category",
        "Description",
        "URL"
    ]
].head(1)

display(selected_product)

recommendations = recommend_similar(
    product_id=test_product_id,
    top_n=5
)

display(recommendations)

Selected Product ID: 2296012


,Product_id,BrandName,Category,Individual_category,Description,URL
0,2296012,Roadster,Bottom Wear,jeans,roadster men navy blue slim fit mid rise clean...,https://www.myntra.com/jeans/roadster/roadster...


,Product_id,BrandName,Category,Individual_category,category_by_Gender,Description,URL,similarity_score
0,12303526,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
1,2524998,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
2,2377247,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
3,2425791,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
4,2525001,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719


In [13]:
recommendations = recommend_similar(
    product_id=2296012,
    top_n=5
)

display(recommendations)

,Product_id,BrandName,Category,Individual_category,category_by_Gender,Description,URL,similarity_score
0,12303526,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
1,2524998,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
2,2377247,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
3,2425791,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719
4,2525001,Roadster,Bottom Wear,jeans,Men,roadster men blue slim fit mid rise clean look...,https://www.myntra.com/jeans/roadster/roadster...,0.980719


In [14]:
if not recommendations.empty:
    display(
        recommendations[
            [
                "Product_id",
                "BrandName",
                "Category",
                "similarity_score",
                "URL"
            ]
        ]
    )

    print(
        "Highest similarity score:",
        round(recommendations["similarity_score"].max(), 4)
    )
else:
    print("No recommendations were returned for this product.")

,Product_id,BrandName,Category,similarity_score,URL
0,12303526,Roadster,Bottom Wear,0.980719,https://www.myntra.com/jeans/roadster/roadster...
1,2524998,Roadster,Bottom Wear,0.980719,https://www.myntra.com/jeans/roadster/roadster...
2,2377247,Roadster,Bottom Wear,0.980719,https://www.myntra.com/jeans/roadster/roadster...
3,2425791,Roadster,Bottom Wear,0.980719,https://www.myntra.com/jeans/roadster/roadster...
4,2525001,Roadster,Bottom Wear,0.980719,https://www.myntra.com/jeans/roadster/roadster...


Highest similarity score: 0.9807
